In [1]:
import os
import sys
import torch
from pathlib import Path

if "__file__" in globals():
    project_root = Path(__file__).resolve().parent.parent
else:
    project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

project_root = project_root.resolve()

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

if Path.cwd() != project_root:
    os.chdir(project_root)

from torch.utils.data import TensorDataset, random_split
from core.config import load_config, print_config
from core.data.loader import setup_dataset, load_dataset
from core.data.dataset import create_dataloaders
from core.data.transforms import create_normalizer_from_data
from core.model.bert import BertForPretraining
from core.training.sampler import create_kde_sampler
from core.training.pretrainer import setup_training
from core.logger import print_data_summary, log_model_summary

In [2]:
print(f"PyTorch version: {torch.__version__}")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    torch.set_float32_matmul_precision("high")
    if torch.cuda.is_bf16_supported():
        print("Bfloat16 is supported and will be used.")
print(f"Using device: {device}")
print(f"Working directory: {os.getcwd()}")

config = load_config("config", config_dir=".")
print_config(config, "Loaded BERT Configuration")

PyTorch version: 2.9.0+cu126
Bfloat16 is supported and will be used.
Using device: cuda
Working directory: /home/jessiez/projects/osu_corpora

--- Loaded BERT Configuration ---
data:
  max_seq_len: 2048
  val_split: 0.1
  max_samples_per_class:
    aim: 2500
    tech: 2500
  min_stars: 4.0
  max_stars: 12.0
model:
  d_model: 512
  n_heads: 8
  n_layers: 6
  dim_feedforward_mult: 4
  dropout: 0.1
  local_attention_window: 256
  cnn_kernel_size: 15
components:
  use_flash_attention: true
  compile_model: true
  compile_mode: default
pretraining:
  db_path: ./data/beatmap_dataset_test/
  batch_size: 8
  num_epochs: 8
  learning_rate: 0.0002
  min_lr: 1.0e-06
  cooldown_type: cosine
  weight_decay: 0.05
  warmup_ratio: 0.1
  stable_ratio: 0.1
  use_amp: true
  checkpoint_dir: ./checkpoints
  grad_clip_norm: 1.0
  gradient_accumulation_steps: 8
  difficulty_loss_weight: 1.0
  mlm_loss_weight: 1.0
  masking_ratio: 0.3
  mean_span_length: 4
  sampling:
    method: kde
    kde_bandwidth: 0.2
 

/home/jessiez/projects/osu_corpora/.venv/lib/python3.12/site-packages/torch/__init__.py:1617: UserWarning: Please use the new API settings to control TF32 behavior, such as torch.backends.cudnn.conv.fp32_precision = 'tf32' or torch.backends.cuda.matmul.fp32_precision = 'ieee'. Old settings, e.g, torch.backends.cuda.matmul.allow_tf32 = True, torch.backends.cudnn.allow_tf32 = True, allowTF32CuDNN() and allowTF32CuBLAS() will be deprecated after Pytorch 2.9. Please see https://pytorch.org/docs/main/notes/cuda.html#tensorfloat-32-tf32-on-ampere-and-later-devices (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:80.)
  _C._set_float32_matmul_precision(precision)


In [3]:
colab_url = 'https://drive.google.com/uc?id=14yvshmHQ069SCjKIa8uBak8aPcyccSqv'
DATASET_PATH = setup_dataset(config['pretraining']['db_path'], colab_url)

print(f"Using database: {DATASET_PATH}")

all_beatmaps_data, difficulty_attributes, loaded_ids = load_dataset(
    DATASET_PATH, 
    max_seq_len=config['data']['max_seq_len'],
    raw_beatmap_path=config['pretraining'].get('raw_beatmap_path', './data/raw')
)

print_data_summary(all_beatmaps_data)

Using database: ./data/beatmap_dataset_test/
Loading raw data from Parquet dataset...
Loading beatmap metadata...
Found metadata for 19656 beatmaps. Processing in chunks of 2000...


Processing Chunks: 100%|██████████| 10/10 [04:04<00:00, 24.50s/it]


Consolidating processed chunks...
Loaded raw feature vectors for 19623 beatmaps.
Calculating difficulty attributes (will use cache if available)...


Calculating Attributes: 100%|██████████| 19/19 [00:00<00:00, 119299.07it/s]


Running final data integrity check...


Validating Tensors: 100%|██████████| 19604/19604 [00:01<00:00, 10059.27it/s]

Finished loading and processing all data.

--- Data Summary ---
Total beatmaps: 19604
Vector dimension: 18
Sequence length - Min: 37, Max: 2048, Avg: 828.0
--------------------


In [4]:
val_size = int(len(all_beatmaps_data) * config['data']['val_split'])
train_size = len(all_beatmaps_data) - val_size

temp_dataset = TensorDataset(torch.arange(len(all_beatmaps_data)))
train_split, val_split = random_split(temp_dataset, [train_size, val_size])

print(f"Data split: {len(train_split.indices)} training, {len(val_split.indices)} validation")

train_data_list = [all_beatmaps_data[i] for i in train_split.indices]
val_data_list = [all_beatmaps_data[i] for i in val_split.indices]

train_attributes = {key: val[train_split.indices] for key, val in difficulty_attributes.items()}
val_attributes = {key: val[val_split.indices] for key, val in difficulty_attributes.items()}

sampler = create_kde_sampler(
    train_attributes['stars'],
    bandwidth=config['pretraining']['sampling']['kde_bandwidth'],
    num_bins=config['pretraining']['sampling'].get('num_bins', 100),
)

normalizer = create_normalizer_from_data(train_data_list, train_attributes)
vector_stats = normalizer.get_vector_stats()

print(f"Vector normalization stats for {len(vector_stats)} fields")

Data split: 17644 training, 1960 validation
Creating optimized KDE sampler with bandwidth=0.2, bins=200...
KDE sampling - Min weight: 0.1720, Max weight: 706.3035
Calculating normalization statistics...

                    NORMALIZATION STATISTICS

--- VECTOR STATISTICS:
------------------------------------------------------------
Field Name             Type         Param 1      Param 2     
------------------------------------------------------------
norm_x                 none         N/A          N/A         
norm_y                 none         N/A          N/A         
delta_x                mean/std     -0.0003      110.0838    
delta_y                mean/std     0.0029       98.6064     
log_time_diff_ms       mean/std     4.8654       0.4985      
bpm                    mean/std     184.9032     37.3167     
notes_per_second       mean/std     8.1736       2.7727      
velocity               mean/std     0.8953       0.8623      
relative_angle         none         N/A        

In [5]:
train_dataloader, val_dataloader = create_dataloaders(
    train_data_list,
    val_data_list,
    train_attributes,
    val_attributes,
    normalizer,
    config, device, sampler
)

print(f"Created dataloaders with batch size: {config['pretraining']['batch_size']}")

sample_batch = next(iter(train_dataloader))
print(f"Sample batch shapes: vectors={sample_batch[0].shape}, mask={sample_batch[1].shape}")
print(f"Sample attributes keys: {list(sample_batch[2].keys())}")

Created dataloaders with batch size: 8
Sample batch shapes: vectors=torch.Size([8, 2048, 18]), mask=torch.Size([8, 2048])
Sample attributes keys: ['stars', 'aim', 'speed', 'slider_factor', 'cs', 'ar', 'slider_multiplier']


In [6]:
model = BertForPretraining.from_config(config, device)
log_model_summary(model)

print("\nRunning a test forward pass with mixed precision (autocast)...")
use_amp_for_test = device.type == "cuda"
with torch.no_grad():
    with torch.amp.autocast(device_type=device.type, dtype=torch.bfloat16, enabled=use_amp_for_test):
        sample_vectors, sample_mask, sample_attrs, sample_cu_seqlens = sample_batch
        
        sample_vectors = sample_vectors.to(device)
        sample_mask = sample_mask.to(device)

        predictions, targets, _ = model(sample_vectors, sample_mask)

        print(f"Prediction output keys: {list(predictions.keys())}")
        print(f"MLM prediction keys: {list(predictions['mlm'].keys())}")
        print(f"Difficulty prediction keys: {list(predictions['difficulty'].keys())}")

print("\nBERT model created and tested successfully!")

Compiling BERT pre-training model with torch.compile...

--- BERT Encoder Information ---
Total Parameters: 33.30M
Model Dimension: 512
Number of Heads: 8
Number of Layers: 6
Flash Attention: True
------------------------------

--- Pre-training Head Information ---
Tasks: Masked Modeling, Difficulty Attribute Prediction
Masking Ratio: 0.3
Model Compiled: True
------------------------------

Running a test forward pass with mixed precision (autocast)...


W1225 13:34:23.052000 416361 .venv/lib/python3.12/site-packages/torch/_inductor/utils.py:1558] [11/0_1] Not enough SMs to use max_autotune_gemm mode


Prediction output keys: ['mlm', 'difficulty']
MLM prediction keys: ['continuous', 'categorical']
Difficulty prediction keys: ['stars', 'aim', 'speed', 'slider_factor', 'cs', 'ar', 'slider_multiplier']

BERT model created and tested successfully!


In [7]:
trainer, checkpoint_manager = setup_training(
    model, train_dataloader, val_dataloader, config, device, normalizer
)

start_epoch = 0
if checkpoint_manager.checkpoint_exists():
    try:
        loaded_epoch, metrics = checkpoint_manager.load_checkpoint(
            model, trainer.optimizer, trainer.scheduler, trainer.scaler, device=device
        )
        start_epoch = loaded_epoch + 1 
        print(f"Loaded checkpoint from epoch {loaded_epoch}, resuming from epoch {start_epoch}")
        print(f"Previous metrics: {metrics}")
    except Exception as e:
        print(f"Could not load checkpoint: {e}")
        print("Starting pretraining from scratch")

print(f"Pretraining setup complete. Starting from epoch {start_epoch}")
print(f"Total epochs: {config['pretraining']['num_epochs']}")

Scheduler: WSD with 220 warmup, 220 stable, 1768 decay steps.
Cooldown type: cosine, Min LR Ratio: 0.0050
PreTrainer initialized - AMP: True, Device: cuda, Grad Accum: 8
Effective batch size: 64
Pretraining setup complete. Starting from epoch 0
Total epochs: 8


In [8]:
print("\nStarting BERT pretraining...")
print(f"BERT Model: {config['model']['n_layers']} layers, {config['model']['d_model']} dimensions")

print(f"Pretraining samples: {len(train_data_list)} base maps")
print(f"Validation samples: {len(val_data_list)} base maps")

metrics_tracker = trainer.train(start_epoch)

print("\nBERT training completed!")


Starting BERT pretraining...
BERT Model: 6 layers, 512 dimensions
Pretraining samples: 17644 base maps
Validation samples: 1960 base maps

--- Starting Training ---
Epochs: 1 to 8
Batch Size: 8
Learning Rate: 0.0002
------------------------------


Epoch 1 [Train]:   0%|          | 0/276 [00:00<?, ?it/s]

Epoch 1 [Validate]:   0%|          | 0/245 [00:00<?, ?it/s]

Epoch 1/8 | Train Loss: 26.7582 (MLM: 9.1629, Diff: 17.5954) | Val Loss: 7.2445 (MLM: 5.9385, Diff: 1.3060) | LR: 2.00e-04 | Time: 343.80s
------------------------------
Validation Metrics
Difficulty MAE:
  aim            : 0.4439
  ar             : 0.2807
  cs             : 0.3741
  slider_factor  : 0.0362
  slider_multiplier: 0.3077
  speed          : 0.3078
  stars          : 0.8035
Continuous Features:
  norm_x         : MAE 0.2506
  norm_y         : MAE 0.3087
  delta_x        : MAE 0.6352
  delta_y        : MAE 0.6325
  log_time_diff_ms: MAE 0.4856
  bpm            : MAE 0.1892
  notes_per_second: MAE 0.4135
  velocity       : MAE 0.4447
  relative_angle : MAE 0.7210
  rhythm_change  : MAE 0.1443
  log_slider_pixel_length: MAE 0.7804
  slider_repeats : MAE 0.1583
  slider_tortuosity: MAE 0.1978
Categorical Features:
  object_type    : Acc 69.61%, Prec 0.6980, Rec 0.6961
  is_new_combo   : Acc 82.29%, Prec 0.7946, Rec 0.8229
  beat_in_measure: Acc 72.42%, Prec 0.7186, Rec 0.7242
 

Epoch 2 [Train]:   0%|          | 0/276 [00:00<?, ?it/s]

...

Epoch 7 [Validate]:   0%|          | 0/245 [00:00<?, ?it/s]

Epoch 7/8 | Train Loss: 3.0441 (MLM: 2.9372, Diff: 0.1069) | Val Loss: 3.3319 (MLM: 2.9941, Diff: 0.3378) | LR: 1.27e-05 | Time: 345.44s
------------------------------
Validation Metrics
Difficulty MAE:
  aim            : 0.1639
  ar             : 0.2948
  cs             : 0.2737
  slider_factor  : 0.0307
  slider_multiplier: 0.2478
  speed          : 0.1475
  stars          : 0.3123
Continuous Features:
  norm_x         : MAE 0.1497
  norm_y         : MAE 0.1720
  delta_x        : MAE 0.3991
  delta_y        : MAE 0.4127
  log_time_diff_ms: MAE 0.1923
  bpm            : MAE 0.0573
  notes_per_second: MAE 0.1472
  velocity       : MAE 0.2926
  relative_angle : MAE 0.5167
  rhythm_change  : MAE 0.0714
  log_slider_pixel_length: MAE 0.6091
  slider_repeats : MAE 0.1429
  slider_tortuosity: MAE 0.1551
Categorical Features:
  object_type    : Acc 85.94%, Prec 0.8603, Rec 0.8594
  is_new_combo   : Acc 88.38%, Prec 0.8767, Rec 0.8838
  beat_in_measure: Acc 85.56%, Prec 0.8551, Rec 0.8556
  t

Epoch 8 [Train]:   0%|          | 0/276 [00:00<?, ?it/s]

Epoch 8 [Validate]:   0%|          | 0/245 [00:00<?, ?it/s]

Epoch 8/8 | Train Loss: 2.9491 (MLM: 2.8562, Diff: 0.0929) | Val Loss: 3.3191 (MLM: 2.9646, Diff: 0.3545) | LR: 1.00e-06 | Time: 378.37s
------------------------------
Validation Metrics
Difficulty MAE:
  aim            : 0.1812
  ar             : 0.3031
  cs             : 0.2804
  slider_factor  : 0.0314
  slider_multiplier: 0.2505
  speed          : 0.1481
  stars          : 0.3316
Continuous Features:
  norm_x         : MAE 0.1478
  norm_y         : MAE 0.1708
  delta_x        : MAE 0.3953
  delta_y        : MAE 0.4095
  log_time_diff_ms: MAE 0.1893
  bpm            : MAE 0.0544
  notes_per_second: MAE 0.1423
  velocity       : MAE 0.2915
  relative_angle : MAE 0.5146
  rhythm_change  : MAE 0.0714
  log_slider_pixel_length: MAE 0.6069
  slider_repeats : MAE 0.1438
  slider_tortuosity: MAE 0.1599
Categorical Features:
  object_type    : Acc 86.01%, Prec 0.8605, Rec 0.8601
  is_new_combo   : Acc 88.47%, Prec 0.8779, Rec 0.8847
  beat_in_measure: Acc 85.81%, Prec 0.8574, Rec 0.8581
  t